# Conflict kill-test for adapter vs zero-shot

Purpose: cheaply test whether simple conflict signals between zero-shot CLIP and a trained adapter predict errors beyond ordinary confidence baselines.

This notebook reconstructs, for each test sample:

- `p_zs(x)`: zero-shot probability from raw CLIP image features and adapter `base_text_features`
- `p_ad(x)`: adapter probability from the loaded trained adapter
- `y`, `pred_ad`, `correct`, `error`

Then it computes:

- conflict: `JS(p_zs, p_ad)`, `1 - cos(p_zs, p_ad)`, argmax disagreement
- confidence baselines: adapter MSP, entropy, margin, logit range, energy, zero-shot confidence, adapter confidence
- BayesAdapter uncertainty if `logits_all` is available
- error-detection AUROC / AURC / E-AURC
- kill-test logistic regression: `Error ~ Entropy_ad + Margin_ad + MSP_ad + JS(p_zs,p_ad)`

Decision heuristic:

- If adding conflict improves held-out AUROC by < 0.01, E-AURC does not improve, or only works on one dataset/adapter, do not continue to evidential/Dirichlet fusion.
- If conflict helps under support label noise / dataset shift / low-entropy-high-conflict samples / multiple adapters, then continue.


In [1]:
from __future__ import annotations

import os
import sys
import json
import math
import random
import importlib
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

# Keep CPU thread use sane inside notebook kernels.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")


'1'

## 1. Configuration

Edit only this cell first. Defaults target the latest small `PP_PROKER_ONEHOT` Caltech101 run style in the repo.


In [ ]:
# If this notebook is at MMRL/notebooks/conflict_kill_test.ipynb,
# then PROJECT_ROOT should resolve to MMRL.
PROJECT_ROOT = Path("..").resolve()
assert (PROJECT_ROOT / "run.py").exists(), f"PROJECT_ROOT seems wrong: {PROJECT_ROOT}"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---- Main experiment knobs ----
METHOD_ALIAS = "CLAP"   # examples: "CLAP", "BayesAdapter", "PP_PROKER_ONEHOT"
DATASET = "caltech101"
SHOTS = 16
SEED = 1
PROTOCOL = "FS"
EXEC_MODE = "cache"
BACKBONE = "ViT-B/16"
DATA_ROOT = "DATASETS"

# Run tag is normally read from the method config in run_plan.sh.
# Set manually here to avoid shelling out.
RUN_TAG_BY_ALIAS = {
    "ZS": "ZS",
    "CLAP": "CLAP",
    "PP_PROKER_ONEHOT": "PP_PROKER_ONEHOT",
    "ECKA": "ECKA",
    "CAPEL": "CAPEL",
    "VNC_CAPEL": "VNC_CAPEL",
    "BayesAdapter": "BAYES_ADAPTER",
    "BAYES_ADAPTER": "BAYES_ADAPTER",
    "TipA": "TipA",
    "TipA-f-": "TipA-f-",
    "CrossModal": "CrossModal",
}

RUN_TAG = RUN_TAG_BY_ALIAS.get(METHOD_ALIAS, METHOD_ALIAS)
BACKBONE_DIR = BACKBONE.replace("/", "-")

# This matches MMRL/run_plan.sh build_outdir() for ClipAdapters aliases.
MODEL_DIR = PROJECT_ROOT / "output_refactor" / "ClipAdapters" / RUN_TAG / PROTOCOL / "fewshot_train" / DATASET / f"shots_{SHOTS}" / BACKBONE_DIR / f"seed{SEED}"

# For eval-only collection artifacts.
OUT_DIR = PROJECT_ROOT / "output_refactor" / "analysis" / "conflict_kill_test" / RUN_TAG / DATASET / f"shots_{SHOTS}" / BACKBONE_DIR / f"seed{SEED}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOAD_EPOCH = None  # None means load model-best if present, else the last model.pth.tar-*.

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_DIR:", MODEL_DIR)
print("OUT_DIR:", OUT_DIR)
print("MODEL_DIR exists:", MODEL_DIR.exists())


PROJECT_ROOT: /root/autodl-tmp/MMRL
MODEL_DIR: /root/autodl-tmp/MMRL/output_refactor/ClipAdapters/CLAP/FS/fewshot_train/caltech101/shots_1/ViT-B-16/seed1
OUT_DIR: /root/autodl-tmp/MMRL/output_refactor/analysis/conflict_kill_test/CLAP/caltech101/shots_1/ViT-B-16/seed1
MODEL_DIR exists: True


In [3]:
METHOD_CONFIG_BY_ALIAS = {
    "ZS": "configs/methods/clip_adapters_zs.yaml",
    "CLAP": "configs/methods/clip_adapters_clap.yaml",
    "PP_PROKER_ONEHOT": "configs/methods/clip_adapters_pp_proker_onehot.yaml",
    "ECKA": "configs/methods/clip_adapters_ecka.yaml",
    "CAPEL": "configs/methods/clip_adapters_capel.yaml",
    "VNC_CAPEL": "configs/methods/clip_adapters_vnccapel.yaml",
    "RANDOM": "configs/methods/clip_adapters_random.yaml",
    "TR": "configs/methods/clip_adapters_tr.yaml",
    "TaskRes": "configs/methods/clip_adapters_tr.yaml",
    "ClipA": "configs/methods/clip_adapters_clipa.yaml",
    "CLIPA": "configs/methods/clip_adapters_clipa.yaml",
    "TipA": "configs/methods/clip_adapters_tipa.yaml",
    "TipA-f-": "configs/methods/clip_adapters_tipa_f.yaml",
    "TipA-F": "configs/methods/clip_adapters_tipa_f.yaml",
    "CrossModal": "configs/methods/clip_adapters_crossmodal.yaml",
    "BayesAdapter": "configs/methods/clip_adapters_bayes.yaml",
    "BAYES_ADAPTER": "configs/methods/clip_adapters_bayes.yaml",
    "BayesAdapter_l2": "configs/methods/clip_adapters_bayes_clap.yaml",
    "BAYES_ADAPTER_l2": "configs/methods/clip_adapters_bayes_clap.yaml",
}

assert METHOD_ALIAS in METHOD_CONFIG_BY_ALIAS, f"Unknown METHOD_ALIAS={METHOD_ALIAS}"

METHOD_CONFIG = METHOD_CONFIG_BY_ALIAS[METHOD_ALIAS]
DATASET_CONFIG = f"configs/datasets/{DATASET}.yaml"
PROTOCOL_CONFIG = "configs/protocols/fs.yaml" if PROTOCOL == "FS" else "configs/protocols/b2n.yaml"
RUNTIME_CONFIG = "configs/runtime/adapter_family.yaml"
LAUNCH_METHOD = "ClipAdapters"

for rel in [METHOD_CONFIG, DATASET_CONFIG, PROTOCOL_CONFIG, RUNTIME_CONFIG]:
    assert (PROJECT_ROOT / rel).exists(), f"Missing config: {PROJECT_ROOT / rel}"

print("METHOD_CONFIG:", METHOD_CONFIG)
print("DATASET_CONFIG:", DATASET_CONFIG)
print("PROTOCOL_CONFIG:", PROTOCOL_CONFIG)
print("RUNTIME_CONFIG:", RUNTIME_CONFIG)


METHOD_CONFIG: configs/methods/clip_adapters_clap.yaml
DATASET_CONFIG: configs/datasets/caltech101.yaml
PROTOCOL_CONFIG: configs/protocols/fs.yaml
RUNTIME_CONFIG: configs/runtime/adapter_family.yaml


## 2. Build trainer and load trained adapter

This follows the project’s `run.py` pattern: import dataset modules, import `trainers.refactor_runner`, build cfg, then build trainer and load checkpoint.


In [ ]:
from dassl.engine import build_trainer
from dassl.utils import set_random_seed, setup_logger

from core.config import setup_cfg
from core.utils import import_optional_modules

import_optional_modules([
    "datasets.oxford_pets", "datasets.oxford_flowers", "datasets.fgvc_aircraft",
    "datasets.dtd", "datasets.eurosat", "datasets.stanford_cars", "datasets.food101",
    "datasets.sun397", "datasets.caltech101", "datasets.ucf101", "datasets.imagenet",
    "datasets.imagenetv2", "datasets.imagenet_sketch", "datasets.imagenet_a", "datasets.imagenet_r",
])

# Required registration side effect, same as run.py.
importlib.import_module("trainers.refactor_runner")

args = SimpleNamespace(
    root=str(PROJECT_ROOT / DATA_ROOT),
    output_dir=str(OUT_DIR),
    dataset_config_file=str(PROJECT_ROOT / DATASET_CONFIG),
    method_config_file=str(PROJECT_ROOT / METHOD_CONFIG),
    protocol_config_file=str(PROJECT_ROOT / PROTOCOL_CONFIG),
    runtime_config_file=str(PROJECT_ROOT / RUNTIME_CONFIG),
    exp_config="",
    method=LAUNCH_METHOD,
    protocol=PROTOCOL,
    exec_mode=EXEC_MODE,
    seed=SEED,
    trainer="RefactorRunner",
    eval_only=True,
    model_dir=str(MODEL_DIR),
    load_epoch=LOAD_EPOCH,
    no_train=True,
    opts=[
        "DATASET.NUM_SHOTS", str(SHOTS),
        "DATASET.SUBSAMPLE_CLASSES", "all",
        "MODEL.BACKBONE.NAME", BACKBONE,
    ],
)

if SEED >= 0:
    set_random_seed(SEED)

cfg = setup_cfg(args)
setup_logger(cfg.OUTPUT_DIR)

trainer = build_trainer(cfg)
trainer.load_model(str(MODEL_DIR), epoch=LOAD_EPOCH)
trainer.set_model_mode("eval")

method = trainer.method
model = method.model
device = trainer.device

print("Loaded method:", getattr(method, "method_name", None))
print("Adapter:", type(model.adapter).__name__, getattr(model.adapter, "initialization_name", None))
print("Device:", device)
print("Num classes:", len(trainer.dm.dataset.classnames))
print("Test samples:", len(trainer.test_loader.dataset))


Loading trainer: RefactorRunner
Loading dataset: Caltech101
Reading split from /root/autodl-tmp/MMRL/DATASETS/caltech-101/split_zhou_Caltech101.json
Loading preprocessed few-shot data from /root/autodl-tmp/MMRL/DATASETS/caltech-101/split_fewshot/shot_1-seed_1.pkl
Building transform_train
+ random resized crop (size=(224, 224), scale=(0.5, 1))
+ random flip
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
---------  ----------
Dataset    Caltech101
# classes  100
# train_x  100
# val      100
# test     2,465
---------  ----------
Using Zero-Shot initialization in Linear Probing
[ClipAdapters] trainable params before cache hooks: {'adapter.prototypes'}
Building transform_train
+ random resize

cache train rep 14/20:   0%|          | 0/1 [00:00<?, ?it/s]        

## 3. Helpers: probabilities, conflict, AUROC/AURC


In [ ]:
EPS = 1e-12

def softmax_np(logits: np.ndarray) -> np.ndarray:
    z = logits - np.max(logits, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.clip(e.sum(axis=1, keepdims=True), EPS, None)

def entropy_np(p: np.ndarray) -> np.ndarray:
    p = np.clip(p, EPS, 1.0)
    return -(p * np.log(p)).sum(axis=1)

def js_divergence_np(p: np.ndarray, q: np.ndarray) -> np.ndarray:
    p = np.clip(p, EPS, 1.0)
    q = np.clip(q, EPS, 1.0)
    p = p / p.sum(axis=1, keepdims=True)
    q = q / q.sum(axis=1, keepdims=True)
    m = 0.5 * (p + q)
    kl_pm = (p * (np.log(p) - np.log(m))).sum(axis=1)
    kl_qm = (q * (np.log(q) - np.log(m))).sum(axis=1)
    return 0.5 * (kl_pm + kl_qm)

def cosine_distance_np(p: np.ndarray, q: np.ndarray) -> np.ndarray:
    num = (p * q).sum(axis=1)
    den = np.linalg.norm(p, axis=1) * np.linalg.norm(q, axis=1)
    return 1.0 - num / np.clip(den, EPS, None)

def margin_np(p: np.ndarray) -> np.ndarray:
    top2 = np.partition(p, kth=-2, axis=1)[:, -2:]
    top2.sort(axis=1)
    return top2[:, 1] - top2[:, 0]

def safe_roc_auc(error: np.ndarray, score_high_means_error: np.ndarray) -> float:
    error = np.asarray(error).astype(int)
    score = np.asarray(score_high_means_error).astype(float)
    mask = np.isfinite(score)
    if mask.sum() < 2 or len(np.unique(error[mask])) < 2:
        return float("nan")
    return float(roc_auc_score(error[mask], score[mask]))

def aurc_from_error_score(error: np.ndarray, score_high_means_error: np.ndarray) -> float:
    """AURC where low score is kept first and high score is rejected first."""
    error = np.asarray(error).astype(float)
    score = np.asarray(score_high_means_error).astype(float)
    mask = np.isfinite(score)
    error = error[mask]
    score = score[mask]
    if len(error) == 0:
        return float("nan")
    order = np.argsort(score)  # most reliable first
    e = error[order]
    risks = np.cumsum(e) / np.arange(1, len(e) + 1)
    return float(np.mean(risks))

def optimal_aurc(error: np.ndarray) -> float:
    error = np.asarray(error).astype(float)
    if len(error) == 0:
        return float("nan")
    order = np.argsort(error)  # correct samples first
    e = error[order]
    risks = np.cumsum(e) / np.arange(1, len(e) + 1)
    return float(np.mean(risks))

def detection_table(df: pd.DataFrame, score_cols: list[str]) -> pd.DataFrame:
    rows = []
    error = df["error"].to_numpy().astype(int)
    opt = optimal_aurc(error)
    for col in score_cols:
        score = df[col].to_numpy(dtype=float)
        auroc = safe_roc_auc(error, score)
        aurc = aurc_from_error_score(error, score)
        rows.append({
            "score": col,
            "auroc": auroc,
            "aurc": aurc,
            "eaurc": aurc - opt if np.isfinite(aurc) and np.isfinite(opt) else np.nan,
        })
    return pd.DataFrame(rows).sort_values(["auroc", "eaurc"], ascending=[False, True])


## 4. Collect per-sample zero-shot and adapter predictions

`p_ad` comes from `model.forward_features(...)`, which uses the adapter and cache logits if applicable.

`p_zs` uses raw image features against `model.adapter.base_text_features`, i.e. the zero-shot CLIP text prototypes already built by the project for the same class order.


In [ ]:
@torch.no_grad()
def collect_predictions(trainer, max_batches: int | None = None):
    trainer.set_model_mode("eval")
    method = trainer.method
    model = method.model
    device = trainer.device
    classnames = list(trainer.dm.dataset.classnames)

    base_text = model.adapter.base_text_features.to(device)
    logit_scale = model.logit_scale.exp()

    rows = []
    zs_logits_all = []
    ad_logits_all = []
    labels_all = []
    bayes_mi_all = []
    bayes_pred_entropy_all = []
    bayes_expected_entropy_all = []
    bayes_prob_var_all = []

    n_test_samples = int(getattr(
        trainer.cfg.CLIP_ADAPTERS,
        "N_TEST_SAMPLES",
        10 if str(getattr(model.adapter, "initialization_name", "")).upper() == "BAYES_ADAPTER" else getattr(trainer.cfg.CLIP_ADAPTERS, "N_SAMPLES", 3),
    ))

    for batch_idx, batch in enumerate(trainer.test_loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        image = batch["img"].to(device)
        label = batch["label"].to(device).long()

        try:
            image_features = model.image_encoder(image.type(model.dtype))
        except Exception:
            image_features = model.image_encoder(image.float())

        # Adapter logits: identical path used by ClipAdaptersModel.forward_eval, but keeps logits_all.
        head = model.forward_features(image_features, n_samples=n_test_samples)
        ad_logits = head["logits"]

        # Zero-shot logits: raw CLIP image features vs original zero-shot text features.
        img_norm = F.normalize(image_features.float(), dim=-1)
        text_norm = F.normalize(base_text.float(), dim=-1)
        zs_logits = img_norm @ text_norm.t() * logit_scale

        logits_all = head.get("logits_all", None)
        if logits_all is not None:
            probs_sbc = torch.softmax(logits_all.float(), dim=-1)  # [S, B, C]
            mean_probs = probs_sbc.mean(dim=0)
            pred_ent = -(mean_probs.clamp_min(EPS) * mean_probs.clamp_min(EPS).log()).sum(dim=-1)
            exp_ent = -(probs_sbc.clamp_min(EPS) * probs_sbc.clamp_min(EPS).log()).sum(dim=-1).mean(dim=0)
            mi = pred_ent - exp_ent
            prob_var = probs_sbc.var(dim=0).mean(dim=-1)
        else:
            b = int(label.shape[0])
            mi = torch.full((b,), float("nan"), device=device)
            pred_ent = torch.full((b,), float("nan"), device=device)
            exp_ent = torch.full((b,), float("nan"), device=device)
            prob_var = torch.full((b,), float("nan"), device=device)

        zs_logits_all.append(zs_logits.detach().cpu())
        ad_logits_all.append(ad_logits.detach().cpu())
        labels_all.append(label.detach().cpu())
        bayes_mi_all.append(mi.detach().cpu())
        bayes_pred_entropy_all.append(pred_ent.detach().cpu())
        bayes_expected_entropy_all.append(exp_ent.detach().cpu())
        bayes_prob_var_all.append(prob_var.detach().cpu())

        paths = batch.get("impath", None) or batch.get("path", None)
        if paths is None:
            paths = [None] * int(label.shape[0])

        offset = sum(x.shape[0] for x in labels_all[:-1])
        for i in range(int(label.shape[0])):
            y = int(label[i].detach().cpu().item())
            rows.append({
                "sample_index": offset + i,
                "path": paths[i] if isinstance(paths, (list, tuple)) else None,
                "y": y,
                "y_name": classnames[y] if 0 <= y < len(classnames) else str(y),
            })

    zs_logits_np = torch.cat(zs_logits_all, dim=0).float().numpy()
    ad_logits_np = torch.cat(ad_logits_all, dim=0).float().numpy()
    labels_np = torch.cat(labels_all, dim=0).long().numpy()
    bayes_mi_np = torch.cat(bayes_mi_all, dim=0).float().numpy()
    bayes_pred_entropy_np = torch.cat(bayes_pred_entropy_all, dim=0).float().numpy()
    bayes_expected_entropy_np = torch.cat(bayes_expected_entropy_all, dim=0).float().numpy()
    bayes_prob_var_np = torch.cat(bayes_prob_var_all, dim=0).float().numpy()

    return {
        "rows": rows,
        "classnames": classnames,
        "zs_logits": zs_logits_np,
        "ad_logits": ad_logits_np,
        "labels": labels_np,
        "bayes_mi": bayes_mi_np,
        "bayes_pred_entropy": bayes_pred_entropy_np,
        "bayes_expected_entropy": bayes_expected_entropy_np,
        "bayes_prob_var": bayes_prob_var_np,
    }

pred = collect_predictions(trainer, max_batches=None)

p_zs = softmax_np(pred["zs_logits"])
p_ad = softmax_np(pred["ad_logits"])
labels = pred["labels"]
classnames = pred["classnames"]

np.savez_compressed(
    OUT_DIR / "conflict_predictions.npz",
    p_zs=p_zs,
    p_ad=p_ad,
    zs_logits=pred["zs_logits"],
    ad_logits=pred["ad_logits"],
    labels=labels,
    classnames=np.array(classnames, dtype=object),
)

print("Collected samples:", len(labels))
print("Saved full probabilities/logits to:", OUT_DIR / "conflict_predictions.npz")


## 5. Build sample table with conflict and baselines


In [ ]:
df = pd.DataFrame(pred["rows"])

pred_zs = p_zs.argmax(axis=1)
pred_ad = p_ad.argmax(axis=1)

df["pred_zs"] = pred_zs
df["pred_ad"] = pred_ad
df["pred_zs_name"] = [classnames[i] for i in pred_zs]
df["pred_ad_name"] = [classnames[i] for i in pred_ad]
df["correct"] = (pred_ad == labels).astype(int)
df["error"] = 1 - df["correct"]

# Conflict signals.
df["js_zs_ad"] = js_divergence_np(p_zs, p_ad)
df["one_minus_cos_zs_ad"] = cosine_distance_np(p_zs, p_ad)
df["argmax_disagree"] = (pred_zs != pred_ad).astype(int)

# Adapter confidence baselines.
df["msp_ad"] = p_ad.max(axis=1)
df["entropy_ad"] = entropy_np(p_ad)
df["margin_ad"] = margin_np(p_ad)
df["logit_range_ad"] = pred["ad_logits"].max(axis=1) - pred["ad_logits"].min(axis=1)

# Energy score: using -logsumexp(logits). Higher means less confident / more likely error.
df["energy_ad"] = -torch.logsumexp(torch.tensor(pred["ad_logits"]).float(), dim=1).numpy()

# Zero-shot confidence baselines.
df["msp_zs"] = p_zs.max(axis=1)
df["entropy_zs"] = entropy_np(p_zs)
df["margin_zs"] = margin_np(p_zs)

# BayesAdapter uncertainty if available; otherwise NaN.
df["bayes_mi"] = pred["bayes_mi"]
df["bayes_pred_entropy"] = pred["bayes_pred_entropy"]
df["bayes_expected_entropy"] = pred["bayes_expected_entropy"]
df["bayes_prob_var"] = pred["bayes_prob_var"]

# Convert confidence metrics to error-detection scores where larger means more likely error.
df["one_minus_msp_ad"] = 1.0 - df["msp_ad"]
df["neg_margin_ad"] = -df["margin_ad"]
df["neg_logit_range_ad"] = -df["logit_range_ad"]
df["one_minus_msp_zs"] = 1.0 - df["msp_zs"]
df["neg_margin_zs"] = -df["margin_zs"]

# Store compact top-k info for inspection. Full p_zs/p_ad are in the NPZ.
def topk_str(p: np.ndarray, k: int = 3) -> list[str]:
    out = []
    for row in p:
        idx = np.argsort(-row)[:k]
        out.append("; ".join(f"{classnames[j]}:{row[j]:.4f}" for j in idx))
    return out

df["top3_zs"] = topk_str(p_zs, k=3)
df["top3_ad"] = topk_str(p_ad, k=3)

sample_csv = OUT_DIR / "conflict_samples.csv"
df.to_csv(sample_csv, index=False)

print("Adapter accuracy:", df["correct"].mean())
print("Adapter error rate:", df["error"].mean())
print("Argmax disagree rate:", df["argmax_disagree"].mean())
print("Saved sample table:", sample_csv)
df.head()


## 6. Error detection: AUROC / AURC / E-AURC


In [ ]:
score_cols = [
    # conflict
    "js_zs_ad",
    "one_minus_cos_zs_ad",
    "argmax_disagree",
    # adapter confidence baselines, transformed so larger means more error-like
    "one_minus_msp_ad",
    "entropy_ad",
    "neg_margin_ad",
    "neg_logit_range_ad",
    "energy_ad",
    # zero-shot confidence baselines
    "one_minus_msp_zs",
    "entropy_zs",
    "neg_margin_zs",
    # BayesAdapter uncertainty, if present
    "bayes_mi",
    "bayes_pred_entropy",
    "bayes_prob_var",
]

metric_df = detection_table(df, score_cols)
metric_csv = OUT_DIR / "conflict_detection_metrics.csv"
metric_df.to_csv(metric_csv, index=False)

print("Saved detection metrics:", metric_csv)
metric_df


## 7. Kill-test: does JS help after controlling confidence?

Main model requested:

`Error(x) ~ Entropy_ad(x) + Margin_ad(x) + MSP_ad(x) + JS(p_zs, p_ad)`

We compare held-out AUROC/AURC against the confidence-only baseline:

`Error(x) ~ Entropy_ad(x) + Margin_ad(x) + MSP_ad(x)`


In [ ]:
BASE_FEATURES = ["entropy_ad", "margin_ad", "msp_ad"]
CONFLICT_FEATURES = BASE_FEATURES + ["js_zs_ad"]

def fit_eval_logreg(df: pd.DataFrame, features: list[str], seed: int = 0, test_size: float = 0.3):
    sub = df[["error"] + features].replace([np.inf, -np.inf], np.nan).dropna().copy()
    y = sub["error"].astype(int).to_numpy()
    X = sub[features].to_numpy(dtype=float)

    if len(np.unique(y)) < 2:
        raise RuntimeError("Need both correct and incorrect samples for logistic regression.")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=seed, stratify=y
    )

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs"),
    )
    clf.fit(X_train, y_train)
    prob = clf.predict_proba(X_test)[:, 1]

    return {
        "features": features,
        "n_train": len(y_train),
        "n_test": len(y_test),
        "test_error_rate": float(y_test.mean()),
        "heldout_auroc": safe_roc_auc(y_test, prob),
        "heldout_aurc": aurc_from_error_score(y_test, prob),
        "heldout_eaurc": aurc_from_error_score(y_test, prob) - optimal_aurc(y_test),
        "model": clf,
    }

base_res = fit_eval_logreg(df, BASE_FEATURES, seed=SEED)
conf_res = fit_eval_logreg(df, CONFLICT_FEATURES, seed=SEED)

summary = pd.DataFrame([
    {k: v for k, v in base_res.items() if k != "model"} | {"model_name": "confidence_only"},
    {k: v for k, v in conf_res.items() if k != "model"} | {"model_name": "confidence_plus_js"},
])

summary["delta_vs_conf_auroc"] = summary["heldout_auroc"] - float(summary.loc[summary.model_name == "confidence_only", "heldout_auroc"].iloc[0])
summary["delta_vs_conf_eaurc"] = summary["heldout_eaurc"] - float(summary.loc[summary.model_name == "confidence_only", "heldout_eaurc"].iloc[0])

kill_csv = OUT_DIR / "kill_test_heldout_summary.csv"
summary.to_csv(kill_csv, index=False)

print("Saved held-out kill-test summary:", kill_csv)
summary


## 8. Cross-validation stability and JS coefficient sign


In [ ]:
def cv_logreg_compare(df: pd.DataFrame, base_features: list[str], conflict_features: list[str], seed: int = 0, n_splits: int = 5):
    sub = df[["error"] + sorted(set(base_features + conflict_features))].replace([np.inf, -np.inf], np.nan).dropna().copy()
    y = sub["error"].astype(int).to_numpy()

    if len(np.unique(y)) < 2:
        raise RuntimeError("Need both correct and incorrect samples for CV.")

    min_class = int(np.bincount(y).min())
    n_splits = max(2, min(n_splits, min_class))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    rows = []
    for fold, (tr, te) in enumerate(skf.split(np.zeros(len(y)), y), start=1):
        y_train, y_test = y[tr], y[te]
        for name, feats in [("confidence_only", base_features), ("confidence_plus_js", conflict_features)]:
            X = sub[feats].to_numpy(dtype=float)
            clf = make_pipeline(
                StandardScaler(),
                LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs"),
            )
            clf.fit(X[tr], y_train)
            prob = clf.predict_proba(X[te])[:, 1]

            row = {
                "fold": fold,
                "model_name": name,
                "auroc": safe_roc_auc(y_test, prob),
                "aurc": aurc_from_error_score(y_test, prob),
                "eaurc": aurc_from_error_score(y_test, prob) - optimal_aurc(y_test),
            }

            lr = clf.named_steps["logisticregression"]
            for feat, coef in zip(feats, lr.coef_[0]):
                row[f"coef_{feat}"] = float(coef)
            rows.append(row)

    cv = pd.DataFrame(rows)
    wide = cv.pivot(index="fold", columns="model_name", values=["auroc", "eaurc"])
    delta = pd.DataFrame({
        "fold": wide.index,
        "delta_auroc": wide[("auroc", "confidence_plus_js")].to_numpy() - wide[("auroc", "confidence_only")].to_numpy(),
        "delta_eaurc": wide[("eaurc", "confidence_plus_js")].to_numpy() - wide[("eaurc", "confidence_only")].to_numpy(),
    })
    return cv, delta

cv_df, cv_delta = cv_logreg_compare(df, BASE_FEATURES, CONFLICT_FEATURES, seed=SEED, n_splits=5)

cv_csv = OUT_DIR / "kill_test_cv_folds.csv"
delta_csv = OUT_DIR / "kill_test_cv_delta.csv"
cv_df.to_csv(cv_csv, index=False)
cv_delta.to_csv(delta_csv, index=False)

print("Saved CV folds:", cv_csv)
print("Saved CV deltas:", delta_csv)
display(cv_df)
display(cv_delta.describe())

js_coef = cv_df.loc[cv_df.model_name == "confidence_plus_js", "coef_js_zs_ad"].dropna()
print("JS coefficient by fold:", js_coef.to_list())
print("JS coefficient positive fraction:", float((js_coef > 0).mean()) if len(js_coef) else np.nan)


## 9. Bootstrap JS coefficient stability

This is a lightweight replacement for p-values. If the bootstrap CI for standardized JS coefficient crosses 0 often, treat JS as unstable.


In [ ]:
def bootstrap_js_coef(df: pd.DataFrame, features: list[str], n_boot: int = 300, seed: int = 0):
    rng = np.random.default_rng(seed)
    sub = df[["error"] + features].replace([np.inf, -np.inf], np.nan).dropna().copy()
    y = sub["error"].astype(int).to_numpy()
    X = sub[features].to_numpy(dtype=float)
    n = len(y)

    rows = []
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        if len(np.unique(y[idx])) < 2:
            continue
        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs"),
        )
        clf.fit(X[idx], y[idx])
        coefs = clf.named_steps["logisticregression"].coef_[0]
        row = {f"coef_{feat}": float(coef) for feat, coef in zip(features, coefs)}
        rows.append(row)
    return pd.DataFrame(rows)

boot = bootstrap_js_coef(df, CONFLICT_FEATURES, n_boot=300, seed=SEED)
boot_csv = OUT_DIR / "kill_test_bootstrap_coefficients.csv"
boot.to_csv(boot_csv, index=False)

coef_col = "coef_js_zs_ad"
if coef_col in boot.columns and len(boot) > 0:
    ci = boot[coef_col].quantile([0.025, 0.5, 0.975])
    print("Saved bootstrap coefficients:", boot_csv)
    print("JS standardized coefficient 2.5/50/97.5% quantiles:")
    print(ci)
    print("Positive fraction:", float((boot[coef_col] > 0).mean()))
else:
    print("No bootstrap coefficients computed.")

boot.describe()


## 10. Inspect low-entropy but high-conflict samples

These are the samples where the direction is most interesting: adapter looks confident, but zero-shot and adapter disagree.


In [ ]:
low_entropy_q = df["entropy_ad"].quantile(0.25)
high_js_q = df["js_zs_ad"].quantile(0.90)

interesting = df[(df["entropy_ad"] <= low_entropy_q) & (df["js_zs_ad"] >= high_js_q)].copy()
interesting = interesting.sort_values(["error", "js_zs_ad"], ascending=[False, False])

interesting_csv = OUT_DIR / "low_entropy_high_conflict_samples.csv"
interesting.to_csv(interesting_csv, index=False)

print("Saved low-entropy high-conflict samples:", interesting_csv)
print("Count:", len(interesting), "Error rate:", interesting["error"].mean() if len(interesting) else np.nan)
interesting[[
    "sample_index", "path", "y_name", "pred_zs_name", "pred_ad_name", "correct",
    "entropy_ad", "msp_ad", "margin_ad", "js_zs_ad", "one_minus_cos_zs_ad",
    "top3_zs", "top3_ad"
]].head(30)


## 11. Automatic decision summary


In [ ]:
base_auroc = float(summary.loc[summary.model_name == "confidence_only", "heldout_auroc"].iloc[0])
conf_auroc = float(summary.loc[summary.model_name == "confidence_plus_js", "heldout_auroc"].iloc[0])
base_eaurc = float(summary.loc[summary.model_name == "confidence_only", "heldout_eaurc"].iloc[0])
conf_eaurc = float(summary.loc[summary.model_name == "confidence_plus_js", "heldout_eaurc"].iloc[0])

delta_auroc = conf_auroc - base_auroc
delta_eaurc = conf_eaurc - base_eaurc  # negative is good
mean_cv_delta_auroc = float(cv_delta["delta_auroc"].mean())
mean_cv_delta_eaurc = float(cv_delta["delta_eaurc"].mean())

js_positive_fraction = float((js_coef > 0).mean()) if len(js_coef) else float("nan")

decision = {
    "method_alias": METHOD_ALIAS,
    "run_tag": RUN_TAG,
    "dataset": DATASET,
    "shots": SHOTS,
    "seed": SEED,
    "n_samples": int(len(df)),
    "adapter_accuracy": float(df["correct"].mean()),
    "heldout_confidence_only_auroc": base_auroc,
    "heldout_confidence_plus_js_auroc": conf_auroc,
    "heldout_delta_auroc": delta_auroc,
    "heldout_confidence_only_eaurc": base_eaurc,
    "heldout_confidence_plus_js_eaurc": conf_eaurc,
    "heldout_delta_eaurc": delta_eaurc,
    "cv_mean_delta_auroc": mean_cv_delta_auroc,
    "cv_mean_delta_eaurc": mean_cv_delta_eaurc,
    "cv_js_coef_positive_fraction": js_positive_fraction,
}

kill = False
reasons = []
if not np.isfinite(delta_auroc) or delta_auroc < 0.01:
    kill = True
    reasons.append("held-out AUROC gain from JS is < 0.01")
if np.isfinite(delta_eaurc) and delta_eaurc >= 0:
    kill = True
    reasons.append("held-out E-AURC did not decrease")
if np.isfinite(js_positive_fraction) and js_positive_fraction < 0.8:
    reasons.append("JS coefficient sign is not stable across CV folds")

decision["preliminary_decision"] = "KILL_OR_PAUSE" if kill else "CONTINUE_TO_MORE_DATASETS_ADAPTERS"
decision["reasons"] = reasons

decision_path = OUT_DIR / "kill_test_decision.json"
with open(decision_path, "w", encoding="utf-8") as f:
    json.dump(decision, f, indent=2, ensure_ascii=False)

print(json.dumps(decision, indent=2, ensure_ascii=False))
print("Saved decision summary:", decision_path)
